In [1]:
from dataclasses import dataclass

@dataclass
class Rule:
    rule_id: str
    conditions: list
    conclusion: str

In [2]:
rules = [
    Rule("R1",
         ["clicking_sound", "engine_not_cranking"],
         "starter_motor_fault"),

    Rule("R2",
         ["battery_light", "dim_headlights"],
         "battery_fault"),

    Rule("R3",
         ["engine_overheating", "coolant_low"],
         "coolant_problem"),

    Rule("R4",
         ["engine_stalls", "fuel_low"],
         "fuel_problem"),

    Rule("R5",
         ["battery_fault", "engine_not_cranking"],
         "starting_system_fault")
]

for r in rules:
    print(f"{r.rule_id}: IF {' AND '.join(r.conditions)} THEN {r.conclusion}")

R1: IF clicking_sound AND engine_not_cranking THEN starter_motor_fault
R2: IF battery_light AND dim_headlights THEN battery_fault
R3: IF engine_overheating AND coolant_low THEN coolant_problem
R4: IF engine_stalls AND fuel_low THEN fuel_problem
R5: IF battery_fault AND engine_not_cranking THEN starting_system_fault


In [3]:
class BackwardChainingEngine:

    def __init__(self, rules, askable_facts=None):
        self.rules = rules
        self.askable_facts = askable_facts or set()
        self.facts = set()
        self.answers = {}
        self.trace = []
        self.current_rule = None

    def find_rules(self, goal):
        return [r for r in self.rules if r.conclusion == goal]

    def ask_fact(self, fact, reason_rule=None):
        if fact in self.answers:
            return self.answers[fact]

        while True:
            question = fact.replace("_", " ").capitalize()

            if reason_rule:
                answer = input(
                    f"\n{question}? (yes/no/why): "
                ).strip().lower()
            else:
                answer = input(
                    f"\n{question}? (yes/no): "
                ).strip().lower()

            if answer == "why" and reason_rule:
                print(
                    f"\nWHY: I need '{question}' because I am "
                    f"trying to prove '{reason_rule.conclusion.replace('_', ' ')}'."
                )
                print(
                    f"Rule {reason_rule.rule_id}: "
                    f"IF {' AND '.join(reason_rule.conditions)} "
                    f"THEN {reason_rule.conclusion}"
                )
                continue

            if answer in ("yes", "no"):
                self.answers[fact] = (answer == "yes")

                if answer == "yes":
                    self.facts.add(fact)

                return answer == "yes"

            print("Please enter yes, no, or why.")

    def prove(self, goal, parent_rule=None):

        # Already known
        if goal in self.facts:
            return True

        # Try rules that can prove the goal
        matching_rules = self.find_rules(goal)

        for rule in matching_rules:
            self.trace.append(
                f"Trying {rule.rule_id}: {rule.conclusion}"
            )

            all_true = True

            for condition in rule.conditions:

                if condition in self.facts:
                    continue

                # If condition itself can be derived, recursively prove it
                if self.find_rules(condition):
                    if not self.prove(condition, rule):
                        all_true = False
                        break

                # Otherwise ask the user
                else:
                    if not self.ask_fact(condition, rule):
                        all_true = False
                        break

            if all_true:
                self.facts.add(goal)
                self.trace.append(
                    f"Proved {goal} using {rule.rule_id}"
                )
                return True

        # If no rule can derive it, ask directly if it is askable
        if goal in self.askable_facts:
            return self.ask_fact(goal, parent_rule)

        return False

    def diagnose(self, hypotheses):
        self.facts = set()
        self.answers = {}
        self.trace = []

        for hypothesis in hypotheses:

            print(
                f"\nTrying hypothesis: "
                f"{hypothesis.replace('_', ' ').upper()}"
            )

            if self.prove(hypothesis):
                print(
                    f"\nDiagnosis: "
                    f"{hypothesis.replace('_', ' ').upper()}"
                )
                return hypothesis

        print("\nNo specific fault could be diagnosed.")
        return None

In [7]:
askable_facts = {
    "clicking_sound",
    "engine_not_cranking",
    "battery_light",
    "dim_headlights",
    "engine_overheating",
    "coolant_low",
    "engine_stalls",
    "fuel_low"
}

engine = BackwardChainingEngine(rules, askable_facts)

hypotheses = [
    "starter_motor_fault"
]

diagnosis = engine.diagnose(hypotheses)


Trying hypothesis: STARTER MOTOR FAULT

WHY: I need 'Clicking sound' because I am trying to prove 'starter motor fault'.
Rule R1: IF clicking_sound AND engine_not_cranking THEN starter_motor_fault

Diagnosis: STARTER MOTOR FAULT


In [8]:
class AutoBackwardEngine(BackwardChainingEngine):

    def __init__(self, rules, known_facts):
        super().__init__(rules)
        self.facts = set(known_facts)
        self.asked = []

    def ask_fact(self, fact, reason_rule=None):
        self.asked.append(fact)

        # Facts not in known_facts are considered false
        return fact in self.facts

    def diagnose_order(self, hypotheses):
        self.asked = []

        for hypothesis in hypotheses:
            if self.prove(hypothesis):
                return hypothesis

        return None

In [15]:
test_facts = {
    "clicking_sound",
    "engine_not_cranking"
}

order_1 = [
    "battery_fault",
    "starter_motor_fault",
    "coolant_problem"
]

order_2 = [
    "starter_motor_fault",
    "battery_fault",
    "coolant_problem"
]

order_3 = [
    "coolant_problem",
    "battery_fault",
    "starter_motor_fault"
]

engine1 = AutoBackwardEngine(rules, test_facts)
diagnosis1 = engine1.diagnose_order(order_1)

engine2 = AutoBackwardEngine(rules, test_facts)
diagnosis2 = engine2.diagnose_order(order_2)

engine3  = AutoBackwardEngine(rules, test_facts)
diagnosis3 = engine3.diagnose_order(order_3)
print("Order 1:", order_1)
print("Questions asked:", engine1.asked)
print("Diagnosis:", diagnosis1)

print("\nOrder 2:", order_2)
print("Questions asked:", engine2.asked)
print("Diagnosis:", diagnosis2)

print("\nOrder 3:", order_3)
print("Questions asked:", engine3.asked)
print("Diagnosis:", diagnosis3)


Order 1: ['battery_fault', 'starter_motor_fault', 'coolant_problem']
Questions asked: ['battery_light']
Diagnosis: starter_motor_fault

Order 2: ['starter_motor_fault', 'battery_fault', 'coolant_problem']
Questions asked: []
Diagnosis: starter_motor_fault

Order 3: ['coolant_problem', 'battery_fault', 'starter_motor_fault']
Questions asked: ['engine_overheating', 'battery_light']
Diagnosis: starter_motor_fault


In [11]:
engine = BackwardChainingEngine(rules, askable_facts)

engine.diagnose(["starter_motor_fault"])


Trying hypothesis: STARTER MOTOR FAULT

WHY: I need 'Clicking sound' because I am trying to prove 'starter motor fault'.
Rule R1: IF clicking_sound AND engine_not_cranking THEN starter_motor_fault

No specific fault could be diagnosed.


In [17]:
summary = pd.DataFrame({
    "Exercise": [
        "Exercise 1",
        "Exercise 2",
        "Exercise 3"
    ],
    "Feature Implemented": [
        "Starter motor fault diagnosis",
        "Hypothesis-order comparison",
        "WHY explanation facility"
    ],
    "Result": [
        "Correctly identifies starter motor fault",
        "Question sequence changes with hypothesis order",
        "Displays rule and goal responsible for question"
    ]
})

display(summary)

,Exercise,Feature Implemented,Result
0,Exercise 1,Starter motor fault diagnosis,Correctly identifies starter motor fault
1,Exercise 2,Hypothesis-order comparison,Question sequence changes with hypothesis order
2,Exercise 3,WHY explanation facility,Displays rule and goal responsible for question
